Q1: How many lesson pages

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

Answer to Q1: There are 72 lesson pages in the dataset

Q2: Indexing and Searching

In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [6]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [7]:
#generating indexed FAQ documents
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [8]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

Answer to Q2: The filename of the 1st result is "01-agentic-rag/lessons/14-agentic-loop.md".

Q3: RAG

In [9]:
# for Q3; modifying RAGBase class to use the new index and search method
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'content': 1.1}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append('Content: ' + doc['content'])
            lines.append('Filename: ' + doc['filename'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage

In [10]:
#How does the agentic loop keep calling the model until it stops?
#index = build_index(documents)
assistant = RAGBase(index, openai_client)

answer, usage = assistant.rag('How does the agentic loop keep calling the model until it stops?')
print("Answer: ", answer)

print("Input Tokens: ", usage.input_tokens)

Answer:  The agentic loop keeps calling the model in a `while True` loop. After each model response, it checks whether the response contains any `function_call` items:

- If there is a function call, the code runs the tool, appends the tool result to the message history, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is simply: **no function calls in the model’s response**.
Input Tokens:  7135


Answer to Q3: Approx 7000 input tokens are used to retrieve this answer.

Q4: Chunking

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
print("Number of chunks: ", len(chunks))

Number of chunks:  295


Answer to Q4: The number of chunks generated is 295.

Q5: RAG with Chunking

In [13]:
# index the chunks
index.fit(chunks)

In [14]:
# point RAG at chunk index
chunky_assistant = RAGBase(index, openai_client)

answer, usage = chunky_assistant.rag('How does the agentic loop keep calling the model until it stops?')
print("Answer: ", answer)

print("Input Tokens: ", usage.input_tokens)


Answer:  It keeps calling the model inside a `while True` loop.

After each model response, the code checks whether there were any `function_call` items:

- if yes, it runs the tool, appends the result, and loops again
- if no, it breaks out of the loop

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: the loop continues until the model returns a message with no tool calls.
Input Tokens:  2318


Answer to Q5: The number of input tokens is approx 3X fewer (from 7000 to 2300) after chunking the documents.

Q6: Turning the RAG with chunked index into an agent

In [15]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [16]:
# this index comes from the chunky index in Q5
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'content': 1.1},
    )

In [17]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [18]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [19]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.
"""

In [20]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [21]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


Answer to Q6: The number of times search is called is 3 in this case, so 4 is the closest answer.